## 15. `ClaudeSDKClient`：多轮与打断

> 来源：[Agent SDK reference - Python](https://code.claude.com/docs/en/agent-sdk/python)

`query()` 每次新开会话；要在同一上下文里连续追问、或中途叫停，用 `ClaudeSDKClient`。

- `async with ClaudeSDKClient(options=...) as client`：推荐用法，退出时自动断开、清理子进程（也可手动 `connect()` / `disconnect()`）。
- `await client.query(prompt)` 发一轮；收消息是一对方法：**`receive_response()`** 收到本轮 `ResultMessage` 即止（日常用它），**`receive_messages()`** 持续收流、不自动停（做常驻监听时用）。再 `client.query(...)` 追问，上下文自动保留（client 内部持有 session ID）。
- `await client.interrupt()`：中途打断正在跑的任务（`query()` 做不到；仅 streaming mode 可用）。**它发的是停止信号，不清空消息缓冲**——被打断任务的剩余消息（收尾是 `subtype="error_during_execution"` 的 `ResultMessage`）还在流里，必须先用 `receive_response()` 排干再发新 query，否则新 query 后第一次收到的是旧任务的消息。下方第二个 cell 演示完整顺序。
- `await client.set_permission_mode(...)` / `await client.set_model(...)`：运行期动态切换权限模式 / 模型。
- `get_server_info()`：查会话的 session ID 与 capabilities；MCP 连接管理：`get_mcp_status()` / `reconnect_mcp_server()` / `toggle_mcp_server()`。

典型形态是把它包成一个会话对象：外层 `while True` 收用户输入，特殊命令映射到 client 方法（`interrupt` → 打断 + 排干，`new` → `disconnect()` 后重新 `connect()` 开新会话），普通输入走 `query()` + `receive_response()`。

In [3]:
import anyio
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
)


async def demo_multiturn():
    options = ClaudeAgentOptions(cwd=".", allowed_tools=["Read", "Glob"], max_turns=4)
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Read the project and summarize the main modules.")
        async for m in client.receive_response():
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, TextBlock):
                        print("A:", b.text)

        # 追问，自动带上一轮上下文
        await client.query("Now focus only on authentication-related files.")
        async for m in client.receive_response():
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, TextBlock):
                        print("A:", b.text)


await demo_multiturn()

# anyio.run(demo_multiturn)

A: I'll explore the project structure to identify and summarize the main modules.
A: Let me read the Jupyter notebook to understand the project:
A: Based on my reading of this Jupyter notebook project about Claude Agent SDK for Python, here's a summary of the main modules:

## Project Overview
This is a comprehensive tutorial notebook titled **"Claude Agent SDK · Python 从 0 到 1"** (Claude Agent SDK Python from 0 to 1) that teaches how to use the Claude Agent SDK.

## Main Modules/Sections

### 1. **Core Concepts & Architecture** (Sections 1-2)
- Explains the fundamental difference between Client SDK and Agent SDK
- Details the agent loop mechanism and why Agent SDK exists
- Clarifies the relationship between Claude Code CLI, Agent SDK, Managed Agents, and Client SDK
- Describes the subprocess architecture and how the agent runtime works

### 2. **Entry Points & Configuration** (Sections 3-5)
- **Installation & Authentication**: Setup guide with API key configuration
- **Two Entry Point

In [ ]:
import asyncio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, ResultMessage


async def demo_interrupt():
    options = ClaudeAgentOptions(allowed_tools=["Bash"])
    async with ClaudeSDKClient(options=options) as client:
        # 起一个长任务，2 秒后打断
        await client.query("Count from 1 to 100 slowly, one number per line")
        await asyncio.sleep(2)
        await client.interrupt()   # ① 发停止信号——注意：不清空消息缓冲

        # ② 排干被打断任务的剩余消息（收尾是 error_during_execution 的 ResultMessage）
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print("interrupted:", message.subtype)

        # ③ 缓冲干净了才能发新任务；跳过②的话，这里第一次收到的是旧任务的消息
        await client.query("Say hello")
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print("new task:", message.subtype)


await demo_interrupt()